In [0]:
# Cargar la tabla raw creada en el notebook anterior
from pyspark.sql.functions import col, sum as spark_sum, count, isnan, when

df = spark.table("wine_quality_raw")

print(f"Total de filas: {df.count()}")
print(f"Total de columnas: {len(df.columns)}")

In [0]:
# Revisar valores nulos en cada columna
null_counts = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])
display(null_counts)

In [0]:
# Detectar y eliminar duplicados
total = df.count()
distinct = df.distinct().count()
duplicates = total - distinct

print(f"Filas totales:    {total}")
print(f"Filas únicas:     {distinct}")
print(f"Duplicados:       {duplicates}")

df_clean = df.dropDuplicates()

In [0]:
# Estadísticas descriptivas básicas
display(df_clean.describe())

In [0]:
# Revisar el rango de 'quality' (debe ser 0-10)
display(df_clean.groupBy("quality").count().orderBy("quality"))

In [0]:
# Crear columna de categoría de calidad
# Esta columna se usará en el análisis y el dashboard
df_clean = df_clean.withColumn(
    "quality_category",
    when(col("quality") >= 7, "Alta")
    .when(col("quality") >= 5, "Media")
    .otherwise("Baja")
)

In [0]:
# Guardar tabla limpia
df_clean.write.format("delta").mode("overwrite").saveAsTable("wine_quality_clean")

print(f"✅ Tabla 'wine_quality_clean' guardada con {df_clean.count()} filas.")
display(df_clean.limit(5))